In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece torch sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("data-final - Sheet1.csv")


In [ ]:
import os
os.environ["HF_TOKEN"] = "your-huggingface-token-here"


In [ ]:
import pandas as pd

df = pd.read_csv("data-final - Sheet1.csv")  # change path if needed
df.head()


,company,quarter,analyst,question,executive,answer
0,JPM,Q4_2025,Glenn Schorr,On the stablecoin issue: given the ABA letter ...,Jeremy Barnum,I'll start by saying you probably know more ab...
1,JPM,Q4_2025,Glenn Schorr,You noted 1.7 million net new checking account...,Jeremy Barnum,"Oh, interesting. So I think what you're saying..."
2,JPM,Q4_2025,Ken Usdin,You mentioned the expense outlook reflects par...,Jeremy Barnum,"Yes, good question. I chose my words carefully..."
3,JPM,Q4_2025,Ken Usdin,You have been comfortable not delivering posit...,Jeremy Barnum,I would anchor my answer on the word output th...
4,JPM,Q4_2025,Erika Najarian,Investors were optimistic about banks’ macro o...,Jamie Dimon,"When you try to guess the macro environment, i..."


In [ ]:
# ADD THIS CELL

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# load once
sim_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
df = pd.read_csv("data-final - Sheet1.csv")
# ADD THIS CELL

def compute_similarity(question, answer):
    q_emb = sim_model.encode([question])
    a_emb = sim_model.encode([answer])
    score = cosine_similarity(q_emb, a_emb)[0][0]
    return round(float(score), 3)


def interpret_similarity(score):
    if score > 0.7:
        return "High alignment (direct)"
    elif score > 0.4:
        return "Moderate alignment (partial)"
    else:
        return "Low alignment (potentially evasive)"

In [ ]:
row = df.iloc[0]   # or any index
question = row["question"]
answer = row["answer"]

similarity_score = compute_similarity(question, answer)
similarity_label = interpret_similarity(similarity_score)

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
!pip install groq pandas


In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd

creds, _ = default()
gc = gspread.authorize(creds)

sheet = gc.open("data-trial").sheet1   # <-- EXACT sheet name
data = sheet.get_all_records()

df = pd.DataFrame(data)
df.shape


(10, 12)

In [ ]:
def classify_qa(question, answer):
    prompt = f"""
You are an expert in financial disclosure analysis.

Analyze the executive answer to the analyst question.

Return:
Return ONLY valid JSON.
Do not include explanations, markdown, or extra text.
Do not include trailing commas.
All strings must use double quotes.

- evasion_type: Direct / Partial / Evasive
- directness_score: 0–5
- relevance_score: 0–5
- specificity_score: 0–5
- uncertainty_markers: list (e.g. "we believe", "hard to predict")
- deflection_markers: list (e.g. narrative expansion, optimism, generalization)

Question:
{question}

Answer:
{answer}

Respond ONLY in valid JSON.
"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content


In [ ]:
import json

outputs = []

for idx in range(len(df)):
    row = df.iloc[idx]

    result = classify_qa(row["question"], row["answer"])
    parsed = json.loads(result)

    outputs.append(parsed)


In [ ]:
import re
import json

def safe_json_parse(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Attempt to extract JSON block
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        else:
            raise ValueError("No valid JSON found")


In [ ]:
outputs = []

for idx in range(len(df)):
    row = df.iloc[idx]

    raw_output = classify_qa(row["question"], row["answer"])
    parsed = safe_json_parse(raw_output)

    outputs.append(parsed)


In [ ]:
df["evasion_type"] = [o["evasion_type"] for o in outputs]
df["directness_score"] = [o["directness_score"] for o in outputs]
df["relevance_score"] = [o["relevance_score"] for o in outputs]
df["specificity_score"] = [o["specificity_score"] for o in outputs]
df["uncertainty_markers"] = [", ".join(o["uncertainty_markers"]) for o in outputs]
df["deflection_markers"] = [", ".join(o["deflection_markers"]) for o in outputs]

df


,company,quarter,analyst,question,executive,answer,evasion_type,directness_score (0–1),relevance_score (0–1),specificity_score (0–1),uncertainty_markers (count),deflection_markers (count),directness_score,relevance_score,specificity_score,uncertainty_markers,deflection_markers
0,JPM,Q4_2025,Glenn Schorr,On the stablecoin issue: given the ABA letter ...,Jeremy Barnum,I'll start by saying you probably know more ab...,Partial,,,,,,2,4,3,I'll start by saying you probably know more ab...,I think Mary Anne is really the expert at this...
1,JPM,Q4_2025,Glenn Schorr,You noted 1.7 million net new checking account...,Jeremy Barnum,"Oh, interesting. So I think what you're saying...",Partial,,,,,,3,5,4,"I think, I would say, I guess, though, clearly","narrative expansion, optimism, generalization"
2,JPM,Q4_2025,Ken Usdin,You mentioned the expense outlook reflects par...,Jeremy Barnum,"Yes, good question. I chose my words carefully...",Partial,,,,,,3,4,2,"market-dependent, volatile, we're optimistic a...","I chose my words carefully, I don't want to br..."
3,JPM,Q4_2025,Ken Usdin,You have been comfortable not delivering posit...,Jeremy Barnum,I would anchor my answer on the word output th...,Direct,,,,,,4,5,4,,"narrative expansion, optimism, generalization"
4,JPM,Q4_2025,Erika Najarian,Investors were optimistic about banks’ macro o...,Jamie Dimon,"When you try to guess the macro environment, i...",Direct,,,,,,4,5,4,"we do not guess outcomes, we operate in the wo...","narrative expansion, optimism"
5,AAPL,Q4_2025,Benjamin Reitzes,"Tim, can you talk a little bit about iPhone in...",Timothy Cook,"Yes. Ben, I was just there. It's incredibly vi...",Direct,,,,,,4,5,4,,
6,AAPL,Q4_2025,Benjamin Reitzes,"And then services, great upside there, a littl...",Kevan Parekh,"Yes, there was no tax-related impact. Our stro...",Direct,,,,,,5,4,4,,"narrative expansion, optimism"
7,AAPL,Q4_2025,Michael Ng,"I just have two as well. First, just to follow...",Kevan Parekh,"Michael, it's Kevan. Thanks for the question. ...",Direct,,,,,,4,5,2,"Those can vary in any given quarter, I wouldn'...","Our services portfolio is very broad, Our stre..."
8,AAPL,Q4_2025,Wamsi Mohan,"Tim, if I can follow up on your comments about...",Timothy Cook,"If you look at the supply constraints, today, ...",Evasive,,,,,,2,4,2,"I'm not going to predict, we're not predicting","narrative expansion, optimism"
9,AAPL,Q4_2025,Wamsi Mohan,"Okay. And then as a follow-up, how do you talk...",Timothy Cook,"he advertising category, which is a combinatio...",Evasive,,,,,,2,4,3,,"I'm dodging the question intentionally, narrat..."


In [ ]:
df["evasion_intensity"] = (
    5 - df["directness_score"]
) + (
    df["uncertainty_markers"].str.count(",") + 1
)

df["is_financial_firm"] = df["company"].isin(["JPM"]).astype(int)

df["is_tech_firm"] = df["company"].isin(["AAPL"]).astype(int)


In [ ]:
df[["company", "directness_score", "uncertainty_markers", "evasion_intensity"]]


,company,directness_score,uncertainty_markers,evasion_intensity
0,JPM,2,I'll start by saying you probably know more ab...,8
1,JPM,3,"I think, I would say, I guess, though, clearly",7
2,JPM,3,"market-dependent, volatile, we're optimistic a...",8
3,JPM,4,,2
4,JPM,4,"we do not guess outcomes, we operate in the wo...",4
5,AAPL,4,,2
6,AAPL,5,,1
7,AAPL,4,"Those can vary in any given quarter, I wouldn'...",3
8,AAPL,2,"I'm not going to predict, we're not predicting",5
9,AAPL,2,,4


In [ ]:
import pandas as pd

df = pd.read_csv("your_file.csv")
df.tail()

FileNotFoundError: [Errno 2] No such file or directory: 'your_file.csv'